# Colab GPU test: Qwen3.5-9B teacher on a local GPU

This notebook asks whether running the teacher on a Colab GPU is a practical alternative to OpenRouter. It runs the **same 30 pilot prompts** (10 per dataset) with thinking **on** and **off**. Then it measures throughput and projects how long the full run would take.

**How to use**
1. Runtime → Change runtime type → **GPU**. The free tier gives a T4; Pro gives an L4 or A100.
2. Run all cells. When asked, upload `samples/pilot_prompts.jsonl` from the repo.
3. Download the two output files and compare them locally with the API pilot:
   `uv run distill analyze colab_open_on.jsonl colab_open_off.jsonl --out reports/colab.md`

The output schema matches the API pipeline. `provider` holds the GPU name and `cost` is 0.

| GPU | Model used | Notes |
|---|---|---|
| A100 40/80 GB | `Qwen/Qwen3.5-9B` (bf16) | same weights as the OpenRouter teacher |
| L4 24 GB | `RedHatAI/Qwen3.5-9B-FP8-dynamic` | 8-bit; very close to bf16 |
| T4 16 GB (free) | `cyankiwi/Qwen3.5-9B-AWQ-4bit` (fp16) | 4-bit, so this is a **lossier teacher**. The T4 has no bf16, and this path is the least certain to work |

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
# vLLM brings its own torch build, so this takes a few minutes.
!pip install -q vllm==0.30.0

In [ ]:
import torch

gpu = torch.cuda.get_device_name(0)
vram_gb = torch.cuda.get_device_properties(0).total_memory / 2**30
capability = torch.cuda.get_device_capability(0)

if vram_gb >= 35:                      # A100 / H100
    MODEL, DTYPE = "Qwen/Qwen3.5-9B", "bfloat16"
elif capability >= (8, 9):             # L4 (FP8-capable)
    MODEL, DTYPE = "RedHatAI/Qwen3.5-9B-FP8-dynamic", "bfloat16"
else:                                  # T4: no bf16 and 16 GB
    MODEL, DTYPE = "cyankiwi/Qwen3.5-9B-AWQ-4bit", "float16"
print(f"{gpu}, {vram_gb:.0f} GB, sm{capability[0]}{capability[1]} -> {MODEL} ({DTYPE})")

MAX_TOKENS = 16384   # the longest reasoning in the API pilot was ~7.4k tokens
SEED = 0
# Same values as config.yaml (Qwen3.5 model card, general tasks).
SAMPLING = {
    True:  dict(temperature=1.0, top_p=0.95, top_k=20, presence_penalty=1.5),   # thinking on
    False: dict(temperature=0.7, top_p=0.8,  top_k=20, presence_penalty=1.5),   # thinking off
}
# Identical to prompts/open_v1.txt, so prompt_hash matches the API pipeline.
TEMPLATE = (
    "### system\n"
    "You are a helpful assistant. Answer in the same language as the user's prompt. End with a clear final answer.\n"
    "### user\n"
    "{prompt}\n"
)

In [ ]:
import json, os

PROMPTS_FILE = "pilot_prompts.jsonl"
if not os.path.exists(PROMPTS_FILE):
    from google.colab import files
    files.upload()   # pick samples/pilot_prompts.jsonl
with open(PROMPTS_FILE, encoding="utf-8") as f:
    prompts = [json.loads(line) for line in f if line.strip()]
print(len(prompts), "prompts")

In [ ]:
from vllm import LLM

llm = LLM(
    model=MODEL,
    dtype=DTYPE,
    max_model_len=MAX_TOKENS + 4096,         # room for the longest prompt
    gpu_memory_utilization=0.92,
    limit_mm_per_prompt={"image": 0, "video": 0},   # text only: skip the vision encoder's memory profile
    enforce_eager=capability < (8, 0),       # saves memory on T4
    seed=SEED,
)
tokenizer = llm.get_tokenizer()

In [ ]:
import hashlib, time
from datetime import datetime, timezone
from vllm import SamplingParams

system, user = TEMPLATE.split("### user\n", 1)
system, user = system.removeprefix("### system\n").strip(), user.strip()
PROMPT_HASH = hashlib.sha256(TEMPLATE.encode("utf-8")).hexdigest()[:16]


def split_reasoning(text, thinking):
    # The chat template already opens <think> when thinking is on, so the output holds only </think>.
    if "</think>" in text:
        reasoning, answer = text.split("</think>", 1)
        return reasoning.replace("<think>", "").strip(), answer.strip(), False
    if thinking:  # ran out of tokens while still thinking
        return text.replace("<think>", "").strip(), "", True
    return "", text.strip(), False


def run(thinking):
    convs = [[{"role": "system", "content": system},
              {"role": "user", "content": user.format(prompt=p["input"])}] for p in prompts]
    params = SamplingParams(**SAMPLING[thinking], max_tokens=MAX_TOKENS, seed=SEED)
    start = time.time()
    outs = llm.chat(convs, params, chat_template_kwargs={"enable_thinking": thinking})
    seconds = time.time() - start

    records = []
    for p, o in zip(prompts, outs):
        out = o.outputs[0]
        reasoning, answer, unclosed = split_reasoning(out.text, thinking)
        reasoning_tokens = len(tokenizer.encode(reasoning, add_special_tokens=False)) if reasoning else 0
        records.append({
            "id": p["id"], "source_dataset": p["source_dataset"], "input": p["input"],
            "reasoning": reasoning, "answer": answer, "finish_reason": out.finish_reason,
            "prompt_tokens": len(o.prompt_token_ids), "reasoning_tokens": reasoning_tokens,
            "answer_tokens": len(out.token_ids) - reasoning_tokens, "think_unclosed": unclosed,
            "provider": f"colab-vllm/{gpu}", "cost": 0.0, "model": MODEL, "mode": "open", "thinking": thinking,
            "prompt_version": "v1", "prompt_hash": PROMPT_HASH, **SAMPLING[thinking], "seed": SEED,
            "max_tokens": MAX_TOKENS, "timestamp": datetime.now(timezone.utc).isoformat(timespec="seconds"),
        })
    path = f"colab_open_{'on' if thinking else 'off'}.jsonl"
    with open(path, "w", encoding="utf-8") as f:
        for r in records:
            f.write(json.dumps(r, ensure_ascii=False) + "\n")
    tokens = sum(len(o.outputs[0].token_ids) for o in outs)
    print(f"thinking={'on' if thinking else 'off'}: {tokens:,} output tokens in {seconds:.0f}s "
          f"= {tokens / seconds:,.0f} tok/s -> {path}")
    return {"tokens": tokens, "seconds": seconds, "records": records}


results = {thinking: run(thinking) for thinking in (True, False)}

In [ ]:
# Eyeball one output per dataset: garbled text here means the model or quantization is broken on this GPU.
for r in results[True]["records"][::10]:
    print(f"=== {r['source_dataset']} | finish={r['finish_reason']} | reasoning {r['reasoning_tokens']} tok, answer {r['answer_tokens']} tok")
    print("PROMPT:   ", r["input"][:200].replace("\n", " "))
    print("REASONING:", r["reasoning"][:300].replace("\n", " "))
    print("ANSWER:   ", r["answer"][:400].replace("\n", " "), "\n")

## Is Colab worth it?
The full-run estimate uses the API pilot: about **190M output tokens with thinking on** (≈ $34 on OpenRouter), and **30M with thinking off** (≈ $6).
The throughput below is **pessimistic**: with only 30 prompts, the GPU sits half idle while the longest answers finish. A full run keeps hundreds of sequences in flight, so real throughput would be higher. Even 2–3× higher, though, it has to beat the API price for your time and Colab's session limits. Free sessions disconnect after about 12 h and have daily GPU quotas.

In [ ]:
FULL_RUN = {True: (190e6, 34), False: (30e6, 6)}   # (output tokens, API USD) estimated from the API pilot
for thinking, r in results.items():
    tok_s = r["tokens"] / r["seconds"]
    total_tokens, api_usd = FULL_RUN[thinking]
    hours = total_tokens / tok_s / 3600
    print(f"thinking {'on ' if thinking else 'off'}: {tok_s:,.0f} tok/s -> full run ≈ {hours:,.0f} GPU-hours "
          f"on this {gpu} (API ≈ ${api_usd})")
finished = [r for res in results.values() for r in res["records"]]
print(f"finish_reason=length: {sum(r['finish_reason'] == 'length' for r in finished)}/{len(finished)}, "
      f"unclosed think: {sum(r['think_unclosed'] for r in finished)}")

In [ ]:
from google.colab import files
files.download("colab_open_on.jsonl")
files.download("colab_open_off.jsonl")